# Steps 4–5: Results Routing & Clinical Follow-Up Pathways

This notebook covers what happens **after** a screening result is returned.

- **Step 4 — Result routing** (`route_cervical_result`, `run_lung_followup`)
  - Cervical cytology: NORMAL → surveillance; ASCUS/LSIL/ASC-H/HSIL → colposcopy (with 20% LTFU check)
  - Cervical HPV-alone: HPV_NEGATIVE → surveillance; HPV_POSITIVE → 40% 1-year repeat / 60% colposcopy (ASCCP triage)
  - Lung Lung-RADS: RADS 0–3 → communicate results → repeat LDCT at set intervals; RADS 4 → biopsy chain

- **Step 5 — Follow-up execution** (`run_colposcopy`, `run_treatment`, `run_lung_followup`)
  - Colposcopy → CIN grade draw (conditional on triggering result) → LEEP / cone / surveillance
  - LTFU checked explicitly at two cervical nodes and five lung nodes

All logic lives in `src/followup.py`. This notebook demonstrates and validates it in isolation.

In [ ]:
import sys, random
sys.path.insert(0, '../src')

import config as cfg
from population import sample_patient
from followup import (
    route_cervical_result,
    run_colposcopy,
    run_treatment,
    run_cervical_followup,
    run_lung_followup,
)
from metrics import initialize_metrics, print_summary
from screening import run_screening_step, get_eligible_screenings

random.seed(cfg.RANDOM_SEED)
print('Imports OK')

## Result Routing — Cervical
Show which pathway each result category triggers.

In [ ]:
from collections import Counter

# All valid cervical result categories across both test types
results_to_test = ['NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_NEGATIVE', 'HPV_POSITIVE']

# 500 trials per result — enough to see the LTFU stochasticity
N_TRIALS = 500
print(f'Cervical result routing (n={N_TRIALS} trials each)\n')
print(f'  {"Result":<18}  {"colposcopy":>12}  {"one_year_repeat":>16}  {"routine_surv":>13}  {"exit (LTFU)":>12}')
print('  ' + '-' * 77)

for result in results_to_test:
    routes = []
    for _ in range(N_TRIALS):
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, result
        routes.append(route_cervical_result(p, 0))
    cnt = Counter(routes)
    print(
        f'  {result:<18}'
        f'  {cnt.get("colposcopy", 0):>12}'
        f'  {cnt.get("one_year_repeat", 0):>16}'
        f'  {cnt.get("routine_surveillance", 0):>13}'
        f'  {cnt.get("exit", 0):>12}'
    )

print('\nExpected:')
print('  NORMAL / HPV_NEGATIVE  → always routine_surveillance')
print('  Abnormal cytology      → ~80% colposcopy, ~20% exit (LTFU)')
print('  HPV_POSITIVE           → ~48% colposcopy, ~32% one_year_repeat, ~20% exit (LTFU)')
print('                           (60/40 colpo/repeat split applied after ~80% pass LTFU check)')

## Colposcopy Result Distribution
Verify CIN grade draws match config probabilities for each triggering result.

In [ ]:
from collections import Counter

# Triggering results that lead to colposcopy (cytology abnormals + HPV_POSITIVE)
triggering_results = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POSITIVE']
N = 1_000

print(f'Colposcopy CIN grade distribution (n={N:,} draws each)\n')
print(f'  {"Trigger":<18}  {"NORMAL":>8}  {"CIN1":>8}  {"CIN2":>8}  {"CIN3":>8}')
print('  ' + '-' * 55)

for trigger in triggering_results:
    counts = Counter()
    for _ in range(N):
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, trigger
        cin = run_colposcopy(p, 0)
        counts[cin] += 1

    expected = cfg.COLPOSCOPY_RESULT_PROBS.get(f'from_{trigger}', {})
    obs_row  = '  '.join(f'{counts.get(g,0)/N*100:5.1f}%' for g in ['NORMAL','CIN1','CIN2','CIN3'])
    exp_row  = '  '.join(f'{expected.get(g,0)*100:5.1f}%' for g in ['NORMAL','CIN1','CIN2','CIN3'])
    print(f'  {trigger:<18}  observed: {obs_row}')
    print(f'  {"":18}  expected: {exp_row}')
    print()

## Full Cervical Follow-Up — End-to-End Trace
Run 10 patients from screening result → colposcopy → treatment/surveillance.
Print each patient's event log.

In [ ]:
from screening import run_screening_step, get_eligible_screenings

random.seed(99)
DAY     = 0
metrics = initialize_metrics()
traced  = []
pid     = 0

# Collect 10 cervical-eligible patients and run the full follow-up pipeline
while len(traced) < 10:
    p = sample_patient(pid, DAY, 'gynecologist', 'outpatient')
    pid += 1
    if 'cervical' not in get_eligible_screenings(p):
        continue
    result = run_screening_step(p, 'cervical', DAY, metrics)
    if result is None:
        continue
    disposition = run_cervical_followup(p, DAY, metrics)
    p.log(DAY, f'DISPOSITION: {disposition}')
    traced.append(p)

# Print each patient's full event log
for p in traced:
    p.print_history()

## Disposition Summary

In [ ]:
print_summary(metrics)

## Lung Follow-Up — Pathway Routing by RADS Category

Each Lung-RADS category triggers a different next step. RADS 0–3 → communicate results → repeat LDCT at set intervals. RADS 4A/4B/4X → communicate → biopsy chain (5 LTFU nodes).

In [ ]:
from collections import Counter

rads_categories = ['RADS_0', 'RADS_1', 'RADS_2', 'RADS_3', 'RADS_4A', 'RADS_4B_4X']
N_TRIALS = 500

print(f'Lung follow-up routing by RADS category (n={N_TRIALS} trials each)\n')

for rads in rads_categories:
    dispositions = []
    for _ in range(N_TRIALS):
        p = sample_patient(0, 0, 'specialist', 'outpatient')
        p.age, p.smoker, p.pack_years, p.lung_result = 60, True, 30, rads
        m = initialize_metrics()
        disposition = run_lung_followup(p, 0, m)
        dispositions.append(disposition)
    cnt = Counter(dispositions)
    parts = ', '.join(f'{k}={v}' for k, v in sorted(cnt.items()))
    print(f'  {rads:<14}  {parts}')

print()
print('Expected:')
print('  RADS_0         → ~90% repeat_ldct_1_3mo, ~10% exit (result not communicated)')
print('  RADS_1/2       → ~90% repeat_ldct_12mo,  ~10% exit')
print('  RADS_3         → ~90% repeat_ldct_6mo,   ~10% exit')
print('  RADS_4A/4B_4X  → ~5–15% reach lung_treated; most exit at various LTFU nodes')